In [9]:
#main loop 

k = 10 # k-fold validation 
item_set = [10,20]
user_set = [10,20]

#create the datasets 

#main 

for n in item_set: 
    for m in user_set:
        n_items = n 
        n_users = m 
        ten = [str(j).zfill(2) for j in list(range(1,k+1))]
        print(ten)
        

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']


In [1]:
import pandas as pd
import numpy as np
n_items = 100
n_users = 300
k = 10 
ten = [str(j).zfill(2) for j in list(range(1,k+1))]
ten

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']

In [7]:
labs = ['test'+i for i in ten]
labs.insert(0,'orig')
url = ""#"https://raw.githubusercontent.com/park61/imputation/main/data/"
url_dict = {}
url_dict["orig"] = url+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
for i in ten:
    url_dict["test{0}".format(i)] = url+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'

In [8]:
print(labs)
for lab in labs:
  print(url_dict[lab])

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
100-by-300_original_matrix.csv
100-by-300_10_fold_test_data01.csv
100-by-300_10_fold_test_data02.csv
100-by-300_10_fold_test_data03.csv
100-by-300_10_fold_test_data04.csv
100-by-300_10_fold_test_data05.csv
100-by-300_10_fold_test_data06.csv
100-by-300_10_fold_test_data07.csv
100-by-300_10_fold_test_data08.csv
100-by-300_10_fold_test_data09.csv
100-by-300_10_fold_test_data10.csv


In [54]:
df_dict = {}
for lab in labs:
  print(lab)
  df = pd.read_csv(url_dict[lab])
  print(lab)
  df_dict[lab] = df.set_index('item_id')

orig
orig
test01
test01
test02
test02
test03
test03
test04
test04
test05
test05
test06
test06
test07
test07
test08
test08
test09
test09
test10
test10


In [55]:
n_row = df_dict["orig"].shape[0]
n_col = df_dict["orig"].shape[1]
Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
sparsity = 1 - num_Obs / (n_row * n_col)
print(sparsity)

0.4202


In [56]:
#@title Run SoftImpute algorithm
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

acc_list = []
rmse_list = []
mad_list = []
time_list = []

df_orig = df_dict["orig"]
df_orig.columns = range(df_orig.shape[1])

mm = df_orig.shape[0]
nn = df_orig.shape[1]

labs_test = labs[1:]

col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
count = 0 

for lab in labs_test:
  count += 1
  print(lab)
  
  df = df_dict[lab]
  import numpy as np
  masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
  #df, maxs = normal1(masked_df)
  #df, maxs = normal2(masked_df)
  #df, maxs, probs = normal3(masked_df)

  # from missingpy import MissForest
  # imputer = MissForest()

  from fancyimpute import SoftImpute
  imputer = SoftImpute()

  start = time_lib.time()
  imputed = pd.DataFrame(imputer.fit_transform(df))

  # Rounding for when no normalization           
  for i in range(mm):
    for j in range(nn):
      x = imputed.iloc[i,j]
      if x < 1:
        imputed.iloc[i,j] = 1
      elif x > col_max[j]:
        imputed.iloc[i,j] = col_max[j]
      else:
        imputed.iloc[i,j] = round(x,0)

  #imputed = unnormal1(imputed, maxs)
  #imputed = unnormal2(imputed, maxs)
  #imputed = unnormal3(imputed, maxs, probs)

  end = time_lib.time()
  time = end - start

  df_orig.index = range(mm)
  imputed.index = range(mm)

  df_orig.columns = range(nn)
  imputed.columns = range(nn)

  #save the result data
  
  if count<10:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
  else:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
    
  #imputed.to_csv(filename)
  diff = df_orig - imputed
  sq_diff = diff ** 2
  abs_diff = abs(diff)

  n_match = 0
  for i in range(mm):
    for j in range(nn):
      if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
        n_match += 1
  acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(acc)  
  rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
  mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(rmse)
  print(mad)
  print(time)
  acc_list.append(acc)
  rmse_list.append(rmse)
  mad_list.append(mad)
  time_list.append(time)

test01
[SoftImpute] Max Singular Value of X_init = 378.567409
[SoftImpute] Iter 1: observed MAE=0.399499 rank=100
[SoftImpute] Iter 2: observed MAE=0.406708 rank=99
[SoftImpute] Iter 3: observed MAE=0.413277 rank=95
[SoftImpute] Iter 4: observed MAE=0.417554 rank=89
[SoftImpute] Iter 5: observed MAE=0.420323 rank=82
[SoftImpute] Iter 6: observed MAE=0.421343 rank=76
[SoftImpute] Iter 7: observed MAE=0.421774 rank=71
[SoftImpute] Iter 8: observed MAE=0.423012 rank=68
[SoftImpute] Iter 9: observed MAE=0.424389 rank=65
[SoftImpute] Iter 10: observed MAE=0.424918 rank=60
[SoftImpute] Iter 11: observed MAE=0.425010 rank=58
[SoftImpute] Iter 12: observed MAE=0.425170 rank=55
[SoftImpute] Iter 13: observed MAE=0.425108 rank=54
[SoftImpute] Iter 14: observed MAE=0.425276 rank=54
[SoftImpute] Iter 15: observed MAE=0.425426 rank=53
[SoftImpute] Iter 16: observed MAE=0.425658 rank=53
[SoftImpute] Iter 17: observed MAE=0.425779 rank=52
[SoftImpute] Iter 18: observed MAE=0.425844 rank=51
[SoftImput

In [57]:
print(sparsity)

print("Accuracy")
for i in acc_list:
  print(i)
print("\nRMSE")
for i in rmse_list:
  print(i)
print("\nMAD")
for i in mad_list:
  print(i)
print("\nTime")
for i in time_list:
  print(i)

0.4202
Accuracy
0.42380678550891315
0.4353076480736055
0.4485336400230017
0.4439332949971248
0.43588269120184014
0.4427832087406556
0.43735632183908046
0.435632183908046
0.44885057471264367
0.4281609195402299

RMSE
0.9565007145952775
0.9296718892903185
0.9312169657914332
0.9237770306194956
0.9112421616512199
0.9162766870452078
0.9109802728222299
0.9278574999588488
0.8995528902176072
0.9306406148578454

MAD
0.6802760207015526
0.6572742955721679
0.6497987349051179
0.6486486486486487
0.6474985623921794
0.6451983898792409
0.6459770114942529
0.6563218390804598
0.6333333333333333
0.6626436781609195

Time
3.2454404830932617
3.138660192489624
3.142754316329956
3.089862823486328
3.0966622829437256
3.0809781551361084
3.093012571334839
3.0921332836151123
3.087235927581787
3.0896573066711426


In [58]:
df = pd.DataFrame(data=[acc_list, rmse_list,mad_list,time_list])
df.T

,0,1,2,3
0,0.423807,0.956501,0.680276,3.245440
1,0.435308,0.929672,0.657274,3.138660
2,0.448534,0.931217,0.649799,3.142754
3,0.443933,0.923777,0.648649,3.089863
4,0.435883,0.911242,0.647499,3.096662
5,0.442783,0.916277,0.645198,3.080978
6,0.437356,0.910980,0.645977,3.093013
7,0.435632,0.927857,0.656322,3.092133
8,0.448851,0.899553,0.633333,3.087236
9,0.428161,0.930641,0.662644,3.089657


: 